In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from supervised.automl import AutoML

In [18]:
# Load CSV
try:
    df = pd.read_csv("../../Datasets/csv/UAVUnderNormalCyberattacks.csv", low_memory=False)
except FileNotFoundError:
    print("Error: The file '../../Datasets/csv/UAVUnderNormalCyberattacks.csv' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [19]:
# Fill missing class labels according to dataset documentation
df['class_filled'] = df['class']
df.loc[0:9425, 'class_filled'] = 'Benign'
df.loc[13717:25388, 'class_filled'] = 'DoS Attack'
df.loc[26363:38369, 'class_filled'] = 'Replay Attack'
df.loc[39344:45027, 'class_filled'] = 'Evil Twin'
df.loc[50502:54783, 'class_filled'] = 'FDI'

# Remove header rows and NaNs, if any
df = df[df['class_filled'].notna()]
df = df[df['class_filled'] != 'class']

print("Unique labels after filling/cleaning:", df['class_filled'].unique())

Unique labels after filling/cleaning: ['Benign' 'DoS Attack' 'Replay Attack' 'Evil Twin' 'FDI']


In [20]:
# Constants
RANDOM_STATE = 1082
TEST_SIZE = 0.2  # Using 80% training, 20% testing split

In [21]:
# Data Preprocessing
# Drop columns not used as features
drop_cols = ['timestamp_c', 'class', 'class_filled']
X = df.drop(columns=[col for col in drop_cols if col in df.columns], errors='ignore')
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

# Encode labels
le = LabelEncoder()
y = le.fit_transform(df['class_filled'])
y_labels = df['class_filled']  # Save original labels for mljar

print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
print("Unique classes after encoding:", np.unique(y))
num_classes = len(np.unique(y))

X_train, X_test, y_train_labels, y_test_labels = train_test_split(
    X, y_labels, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_labels
)

Label mapping: {'Benign': 0, 'DoS Attack': 1, 'Evil Twin': 2, 'FDI': 3, 'Replay Attack': 4}
Unique classes after encoding: [0 1 2 3 4]


In [22]:
# Build and train model using mljar-supervised
automl = AutoML(
    mode="Explain",  # Use Explain mode to get feature importance and other insights
    algorithms=['Baseline', 'CatBoost', 'Decision Tree', 'Extra Trees', 'Nearest Neighbors', 'LightGBM', 'Linear', 'Neural Network', 'Random Forest', 'Xgboost'],  # All classification algorithms
    ml_task="multiclass_classification",
    results_path="Results",
    total_time_limit=60*30,  # 30 minutes time limit
    random_state=RANDOM_STATE,
    validation_strategy={
        "validation_type": "kfold",
        "k_folds": 5,
        "shuffle": True,
        "random_seed": RANDOM_STATE
    }
)
# Train the model
automl.fit(X_train, y_train_labels)

Linear algorithm was disabled.
AutoML directory: mljar_uav_results
The task is multiclass_classification with evaluation metric logloss
AutoML will use algorithms: ['Baseline', 'CatBoost', 'Decision Tree', 'Extra Trees', 'Nearest Neighbors', 'LightGBM', 'Neural Network', 'Random Forest', 'Xgboost']
AutoML will ensemble available models
AutoML steps: ['simple_algorithms', 'default_algorithms', 'ensemble']
* Step simple_algorithms will try to check up to 2 models
1_Baseline logloss 1.539375 trained in 1.91 seconds
2_DecisionTree logloss 0.535386 trained in 33.51 seconds
* Step default_algorithms will try to check up to 7 models
3_Default_LightGBM logloss 0.120963 trained in 537.42 seconds
4_Default_Xgboost logloss 0.129057 trained in 145.27 seconds
5_Default_CatBoost logloss 0.148095 trained in 185.28 seconds
6_Default_NeuralNetwork logloss 0.44349 trained in 40.13 seconds
7_Default_RandomForest logloss 0.515509 trained in 21.7 seconds
8_Default_ExtraTrees logloss 0.537698 trained in 15.

AutoML(algorithms=['Baseline', 'CatBoost', 'Decision Tree', 'Extra Trees',
                   'Nearest Neighbors', 'LightGBM', 'Linear', 'Neural Network',
                   'Random Forest', 'Xgboost'],
       ml_task='multiclass_classification', random_state=1082,
       results_path='mljar_uav_results', total_time_limit=1800,
       validation_strategy={'k_folds': 5, 'random_seed': 1082, 'shuffle': True,
                            'validation_type': 'kfold'})

In [23]:
# Predict and evaluate
predictions = automl.predict_all(X_test)
print(predictions.head())

print("Test accuracy:", accuracy_score(y_test_labels, predictions["label"]))
print("Classification report:\n", classification_report(y_test_labels, predictions["label"]))
print("Confusion matrix:\n", confusion_matrix(y_test_labels, predictions["label"]))

   prediction_Benign  prediction_DoS Attack  prediction_Evil Twin  \
0       3.105439e-06               0.068992          1.714807e-06   
1       8.070479e-07               0.985466          8.272702e-07   
2       9.999899e-01               0.000009          1.487318e-09   
3       8.628163e-02               0.909926          1.590250e-05   
4       9.992864e-01               0.000101          3.302867e-08   

   prediction_FDI  prediction_Replay Attack          label  
0    8.084486e-07                  0.931003  Replay Attack  
1    4.133773e-07                  0.014532     DoS Attack  
2    4.027647e-10                  0.000001         Benign  
3    7.570851e-06                  0.003769     DoS Attack  
4    1.541835e-08                  0.000612         Benign  
Test accuracy: 0.9619224518226144
Classification report:
                precision    recall  f1-score   support

       Benign       0.93      0.99      0.96      1885
   DoS Attack       0.96      0.94      0.95      